
#### 🌟 主題：籃球賽事播

**🎯 功能說明：**
1. 增加賽況輸入欄位，比分、球隊名稱、關鍵事件
2. Gradio 呈現：三個欄位：初稿播報文案、修改建議、優化後播報文案




#### 1. 讀入你的金鑰

請依你使用的服務, 決定讀入哪個金鑰

In [1]:
import os
from google.colab import userdata

In [2]:
#使用Groq
api_key = userdata.get('Groq')
if not api_key:
    raise ValueError("找不到 Groq API Key：請在 Colab 左側「秘密」新增名稱為 Groq 的金鑰。")
os.environ['GROQ_API_KEY'] = api_key
provider = "groq"
model = "openai/gpt-oss-20b"

In [3]:
!pip install aisuite[all]

### 2. 基本的設定

In [4]:
import aisuite as ai

In [5]:
provider_writer = "groq"
model_writer="openai/gpt-oss-20b"

provider_reviewer = "groq"
model_reviewer = "openai/gpt-oss-20b"

#provider_reviewer = "openai"
#model_reviewer = "gpt-4o"

標準回應函式

In [6]:
def reply(system="請用台灣習慣的中文回覆。",
          prompt="hi",
          provider="groq",
          model="openai/gpt-oss-20b"
          ):

    client = ai.Client()

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt}
    ]


    response = client.chat.completions.create(model=f"{provider}:{model}", messages=messages)

    return response.choices[0].message.content

####  3. 人物設定

In [7]:
system_writer = "你是一位活潑、有感染力的籃球賽事播報主播，專業且有熱情，善於用台灣習慣的中文描繪比賽精彩瞬間。請用第一人稱，讓觀眾感受到現場氣氛，有時加點幽默和 emoji。"
system_reviewer = "你是一位文案專家，擅長讓體育播報文字更口語化、生活化，讓文字生動自然。請針對以下播報文案給出具體修改建議，使用台灣習慣中文。"

In [8]:
def reflect_basketball_report(team_a, score_a, team_b, score_b, highlight):
    # Step1:生成初稿播報文案
    prompt = (
        f"請撰寫一段籃球比賽播報文字，內容包含：\n"
        f"隊伍A：{team_a}，得分：{score_a}\n"
        f"隊伍B：{team_b}，得分：{score_b}\n"
        f"比賽亮點：{highlight}\n"
        f"請用生動且帶熱情的主播口吻撰寫。"
    )
    first_version = reply(system_writer, prompt, provider=provider, model=model_writer)

    # Step2:文案審稿給建議
    suggestion = reply(system_reviewer, first_version, provider=provider, model=model_reviewer)

    # Step3:根據建議再改寫一次
    second_prompt = (
        f"這是我剛剛寫的播報文案：\n{first_version}\n\n"
        f"這是修改建議：\n{suggestion}\n\n"
        f"請根據建議改寫文案，讓播報更流暢自然，台灣用語，且只輸出改好的文案。"
    )
    second_version = reply(system_writer, second_prompt, provider=provider, model=model_writer)

    return first_version, suggestion, second_version

### 4. 用 Gradio 打造你的籃球播報生成系統

In [9]:
!pip install gradio

In [10]:
import gradio as gr

In [11]:
import os
from google.colab import userdata
import gradio as gr
import aisuite as ai


api_key = userdata.get('Groq')
if not api_key:
    raise ValueError("找不到 Groq API Key")
os.environ['GROQ_API_KEY'] = api_key

provider = "groq"
model = "openai/gpt-oss-20b"

provider_writer = "groq"
model_writer="openai/gpt-oss-20b"

provider_reviewer = "groq"
model_reviewer = "openai/gpt-oss-20b"

system_writer = "你是一位活潑、有感染力的籃球賽事播報主播，專業且有熱情，善於用台灣習慣的中文描繪比賽精彩瞬間。請用第一人稱，讓觀眾感受到現場氣氛，有時加點幽默和 emoji。"
system_reviewer = "你是一位文案專家，擅長讓體育播報文字更口語化、生活化，讓文字生動自然。請針對以下播報文案給出具體修改建議，使用台灣習慣中文。"

def reply(system="請用台灣習慣的中文回覆。",
          prompt="hi",
          provider_arg="groq",
          model_arg="openai/gpt-oss-20b"
          ):

    client = ai.Client()

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt}
    ]


    response = client.chat.completions.create(model=f"{provider_arg}:{model_arg}", messages=messages)

    return response.choices[0].message.content

def reflect_basketball_report(team_a, score_a, team_b, score_b, highlight):
    # Step1:生成初稿播報文案
    prompt = (
        f"請撰寫一段籃球比賽播報文字，內容包含：\n"
        f"隊伍A：{team_a}，得分：{score_a}\n"
        f"隊伍B：{team_b}，得分：{score_b}\n"
        f"比賽亮點：{highlight}\n"
        f"請用生動且帶熱情的主播口吻撰寫。"
    )

    first_version = reply(system_writer, prompt, provider_arg=provider, model_arg=model_writer)

    # Step2:文案審稿給建議
    suggestion = reply(system_reviewer, first_version, provider_arg=provider, model_arg=model_reviewer)

    # Step3:根據建議再改寫一次
    second_prompt = (
        f"這是我剛剛寫的播報文案：\n{first_version}\n\n"
        f"這是修改建議：\n{suggestion}\n\n"
        f"請根據建議改寫文案，讓播報更流暢自然，台灣用語，且只輸出改好的文案。"
    )
    second_version = reply(system_writer, second_prompt, provider_arg=provider, model_arg=model_writer)

    return first_version, suggestion, second_version

with gr.Blocks() as demo:
    gr.Markdown("### 🏀 籃球賽事播報文案生成與優化")
    with gr.Row():
        team_a = gr.Textbox(label="隊伍 A 名稱", placeholder="輸入隊伍名稱")
        score_a = gr.Number(label="隊伍 A 得分", value=0)
        team_b = gr.Textbox(label="隊伍 B 名稱", placeholder="輸入隊伍名稱")
        score_b = gr.Number(label="隊伍 B 得分", value=0)
    highlight = gr.Textbox(label="比賽亮點（關鍵事件、精彩瞬間）", lines=3, placeholder="輸入內容")

    btn = gr.Button("生成播報文案 & 修正建議")

    with gr.Row():
        out1 = gr.Textbox(label="📝 初稿播報文案", lines=7)
        out2 = gr.Textbox(label="🔍 修改建議", lines=5)
        out3 = gr.Textbox(label="✨ 優化後播報文案", lines=7)

    btn.click(
        reflect_basketball_report,
        inputs=[team_a, score_a, team_b, score_b, highlight],
        outputs=[out1, out2, out3]
    )

In [12]:
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7634de3f6a03ce66b5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://7634de3f6a03ce66b5.gradio.live
